# Занятие 10. Гомография и масштабирование

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026
Лекция-опора: L10 «Гомография и масштабирование изображения» · 90 минут в классе (последние 20 — клиника ДЗ 2 / мост к ДЗ 3) · ведёт ассистент

---

Сегодня вычислим **гомографию** по четырём парам точек через **DLT** (Direct Linear Transform) и проверим, что она совпадает с `cv2.findHomography`. Потом проведём **аудит ресайзера**: покажем числом, что цикл ×2 → ×1/2 не возвращает оригинал, что два шага ×1/2 ≠ один шаг ×1/4, и что порядок уменьшения по осям влияет на результат. В третьей части реализуем **согласованное уменьшение** — фильтр Гаусса перед прореживанием.

Заготовка начинается на слайдах 19 и 46 лекции («Как вычислить гомографию» и «Тесты: цикл, композиция, поворот»).

| Минуты | Что происходит |
|--------|----------------|
| 0–12 | Ассистент разбирает опорный пример: гомография и три провала ресайзера |
| 12–58 | Вы делаете TODO 1–4 |
| 58–67 | Разбор решения |
| 67–87 | Клиника: ДЗ 2 финал + мост к ДЗ 3 (панорамы) |
| 87–90 | Зачёт |

Ничего сдавать не нужно: зачёт ставится в классе по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.
**Занятие 11 — модель камеры и калибровка.**

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Фото здания ИУ `lectures/L10-scaling/demo/iu-building.jpg` (1920 × 1279) читается в сером;
кроп региона с текстурой 256 × 256 тот же, что в занятиях 5–9. Для гомографии создаётся повёрнутый
и масштабированный кроп с **известной** гомографией.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена.

In [ ]:
LECTURE_DIR = "lectures/L10-scaling/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,
                  "../../../iu-intro-cv/" + LECTURE_DIR,
                  "data"]
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(700, 956), slice(1100, 1356))


def synthetic_photo(h=1279, w=1920):
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[700:956:16, 1100:1356] -= 70
    img[700:956, 1100:1356:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), d
    try:
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        return synthetic_photo(), "синтетика"


photo, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8
crop1 = photo[CROP].astype(np.float64)

h, w = crop1.shape[:2]
theta_true = np.radians(15)
s_true = 0.85
tx, ty = w * 0.08, h * 0.06
H_true = np.array([
    [s_true * np.cos(theta_true), -s_true * np.sin(theta_true), tx],
    [s_true * np.sin(theta_true),  s_true * np.cos(theta_true), ty],
    [0, 0, 1]
], np.float64)
crop2 = cv2.warpPerspective(crop1, H_true, (w, h), borderMode=cv2.BORDER_REPLICATE)

src_pts = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], np.float64)
dst_pts = cv2.perspectiveTransform(src_pts.reshape(1, -1, 2), H_true).reshape(-1, 2)

print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· кроп", crop1.shape)
print("истинная гомография: поворот %.0f°, масштаб %.2f, сдвиг (%.1f, %.1f)" % (
    np.rad2deg(theta_true), s_true, tx, ty))
viz.grid({"кроп 1 (исходный)": crop1 / 255, "кроп 2 (поворот 15°, ×0.85)": crop2 / 255}, cols=2, size=3.4)

## 2. Опорный пример — разбирает ассистент

Здесь всё написано, задача — **понять каждую строку**.

**Гомография** (слайды 4, 19): матрица 3×3, отображающая точки одного изображения в точки другого.
Для её вычисления нужно минимум **4 пары** соответствующих точек.

**Три провала ресайзера** (слайд 46): стандартные библиотеки не удовлетворяют базовым алгебраическим свойствам.

In [ ]:
print("Истинная гомография:")
print(np.array2string(H_true, precision=4, suppress_small=True))
print("\n4 пары точек:")
for i in range(4):
    print("  (%6.1f, %6.1f) → (%6.1f, %6.1f)" % (*src_pts[i], *dst_pts[i]))

img_test = crop1
h_img, w_img = img_test.shape[:2]
rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2))

up = cv2.resize(img_test, (w_img * 2, h_img * 2), interpolation=cv2.INTER_LINEAR)
back = cv2.resize(up, (w_img, h_img), interpolation=cv2.INTER_LINEAR)
print("\nТест цикла (×2 → ×1/2, билинейная): RMSE = %.1e" % rmse(img_test, back))

s1 = cv2.resize(img_test, (w_img // 2, h_img // 2), interpolation=cv2.INTER_LINEAR)
s2 = cv2.resize(s1, (w_img // 4, h_img // 4), interpolation=cv2.INTER_LINEAR)
d4 = cv2.resize(img_test, (w_img // 4, h_img // 4), interpolation=cv2.INTER_LINEAR)
print("Тест композиции (×1/2 ×1/2 vs ×1/4): RMSE = %.1e" % rmse(s2, d4))

hw = cv2.resize(cv2.resize(img_test, (w_img // 2, h_img), interpolation=cv2.INTER_LINEAR),
                (w_img // 2, h_img // 2), interpolation=cv2.INTER_LINEAR)
wh = cv2.resize(cv2.resize(img_test, (w_img, h_img // 2), interpolation=cv2.INTER_LINEAR),
                (w_img // 2, h_img // 2), interpolation=cv2.INTER_LINEAR)
print("Тест коммутативности (Ш→В vs В→Ш): RMSE = %.1e" % rmse(hw, wh))

### Вопрос, на который отвечаем вслух

Почему для гомографии нужно минимум 4 пары точек, а для аффинного преобразования — 3?
(Ответ — слайд 20: аффинное имеет 6 степеней свободы, проективное — 8.)

---

## TODO 1 — гомография через DLT (≈ 12 минут)

Слайды 19, 16. **DLT**: каждая пара точек даёт два линейных уравнения на элементы матрицы H.

Напишите `compute_homography_dlt(src_pts, dst_pts)` → матрица H (3×3), нормированная так что `H[2,2] = 1`.

Скелет — четыре шага:

1. привести точки к `float64`;
2. для каждой пары построить две строки матрицы A;
3. SVD: решение — последняя строка `Vt`;
4. нормировать: `H /= H[2, 2]`.

> **Подсказка.** Строки матрицы A для пары (x, y) → (u, v):
> `[-x, -y, -1, 0, 0, 0, u*x, u*y, u]` и `[0, 0, 0, -x, -y, -1, v*x, v*y, v]`.

In [ ]:
def compute_homography_dlt(src_pts, dst_pts):
    """DLT: гомография по 4+ парам точек → H (3×3), нормирована так что H[2,2] = 1."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10–12 строк: привести к float64; построить A (2N × 9); SVD; последняя строка Vt; reshape; нормировка
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


H_mine = compute_homography_dlt(src_pts, dst_pts)
assert H_mine.shape == (3, 3), "гомография должна быть матрицей 3×3"
assert abs(H_mine[2, 2] - 1.0) < 1e-10, "H[2,2] должна быть нормирована к 1"
H_mine_norm = H_mine / np.linalg.norm(H_mine)
H_true_norm = H_true / np.linalg.norm(H_true)
d = min(np.abs(H_mine_norm - H_true_norm).max(), np.abs(H_mine_norm + H_true_norm).max())
assert d < 1e-6, "DLT должна восстановить истинную гомографию (ошибка %.2e)" % d
H_cv, _ = cv2.findHomography(src_pts, dst_pts, method=0)
d_cv = min(np.abs(H_mine / H_mine[2, 2] - H_cv / H_cv[2, 2]).max(),
           np.abs(H_mine / H_mine[2, 2] + H_cv / H_cv[2, 2]).max())
assert d_cv < 1e-4, "DLT должна совпадать с cv2.findHomography (ошибка %.2e)" % d_cv
print("TODO 1 ✔ · ошибка против истины: %.2e · против cv2.findHomography: %.2e" % (d, d_cv))

### Применение гомографии: выравнивание кадра (слайд 19)

Применим гомографию, чтобы вернуть повёрнутый кроп к оригинальному виду.

In [ ]:
H_inv = np.linalg.inv(H_mine)
restored = cv2.warpPerspective(crop2, H_inv, (w, h))
margin = 20
err = np.abs(crop1[margin:-margin, margin:-margin] - restored[margin:-margin, margin:-margin]).mean()
print("средняя ошибка выравнивания (внутренняя часть): %.2f кодов" % err)
print("PSNR выравнивания: %.1f дБ" % metrics.psnr(
    np.clip(crop1[margin:-margin, margin:-margin], 0, 255).astype(np.uint8),
    np.clip(restored[margin:-margin, margin:-margin], 0, 255).astype(np.uint8)))
viz.grid({"повёрнутый кадр": crop2 / 255, "после выравнивания": restored / 255,
          "оригинал": crop1 / 255}, cols=3, size=3)

---

## TODO 2 — тест цикла ресайзера (≈ 12 минут)

Слайд 46. **Цикл ×2 → ×1/2**: увеличить в 2 раза, затем уменьшить обратно — результат не совпадает с оригиналом.

Напишите `resize_cycle_test(img, scale=2.0, method=cv2.INTER_LINEAR)` → RMSE между оригиналом и результатом цикла.

Скелет — три шага:

1. привести кадр к `float64`;
2. увеличить в `scale` раз;
3. уменьшить обратно; посчитать RMSE.

> **Подсказка.** `cv2.resize(img, (w_new, h_new), interpolation=method)`.
> Размеры: `int(round(h * scale))`, `int(round(w * scale))`.

In [ ]:
def resize_cycle_test(img, scale=2.0, method=cv2.INTER_LINEAR):
    """Тест цикла: увеличить в scale раз, затем уменьшить обратно. Вернуть RMSE."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: привести к float64; размеры; resize вверх; resize вниз; RMSE; return
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


rmse_cycle = resize_cycle_test(crop1, scale=2.0, method=cv2.INTER_LINEAR)
assert rmse_cycle > 0, "цикл ×2 → ×1/2 не может вернуть оригинал точно (для стандартных методов)"
assert rmse_cycle < 5.0, "ошибка цикла должна быть малой (сейчас %.3f)" % rmse_cycle
assert rmse_cycle > 1e-4, "ошибка слишком мала — проверьте, что цикл действительно применяется"
print("TODO 2 ✔ · RMSE цикла ×2 → ×1/2 (билинейная): %.2e" % rmse_cycle)

print("\nСравнение методов интерполяции (тест цикла ×2 → ×1/2):")
for name, method in [("ближайший сосед", cv2.INTER_NEAREST),
                     ("билинейная", cv2.INTER_LINEAR),
                     ("бикубическая", cv2.INTER_CUBIC),
                     ("Lanczos", cv2.INTER_LANCZOS4)]:
    err = resize_cycle_test(crop1, scale=2.0, method=method)
    print("  %-20s RMSE = %.2e" % (name, err))

### Вопрос, на который отвечаем вслух

Почему цикл ×2 → ×1/2 не возвращает оригинал? Какой метод интерполяции даёт наименьшую ошибку?

---

## TODO 3 — тест композиции и коммутативности (≈ 12 минут)

Слайд 46. **Композиция**: два шага ×1/2 ≠ один шаг ×1/4. **Коммутативность**: порядок уменьшения по осям влияет на результат.

Напишите `resize_composition_test(img, method=cv2.INTER_LINEAR)` → `(comp_err, comm_err)`.

Скелет — три шага:

1. **Композиция**: ×1/2 дважды против ×1/4; посчитать RMSE;
2. **Коммутативность**: ширина→высота против высота→ширина; посчитать RMSE;
3. вернуть `(comp_err, comm_err)`.

> **Подсказка.** Для композиции: `step1 = resize(img, w//2, h//2)`, `step2 = resize(step1, w//4, h//4)`,
> `direct = resize(img, w//4, h//4)`.

In [ ]:
def resize_composition_test(img, method=cv2.INTER_LINEAR):
    """Тест композиции и коммутативности ресайзера. Вернуть (comp_err, comm_err)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10–12 строк: привести к float64; композиция (×1/2 ×1/2 vs ×1/4); коммутативность (Ш→В vs В→Ш); RMSE; return
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


comp_err, comm_err = resize_composition_test(crop1, method=cv2.INTER_LINEAR)
assert comp_err > 0, "композиция не может быть точной для стандартных методов"
assert comm_err > 0, "коммутативность не может быть точной для стандартных методов"
assert comp_err < 2.0, "ошибка композиции должна быть малой (сейчас %.3f)" % comp_err
assert comm_err < 2.0, "ошибка коммутативности должна быть малой (сейчас %.3f)" % comm_err
print("TODO 3 ✔ · композиция (×1/2 ×1/2 vs ×1/4): %.2e · коммутативность (Ш→В vs В→Ш): %.2e" % (comp_err, comm_err))

print("\nСравнение методов интерполяции:")
for name, method in [("ближайший сосед", cv2.INTER_NEAREST),
                     ("билинейная", cv2.INTER_LINEAR),
                     ("бикубическая", cv2.INTER_CUBIC),
                     ("Lanczos", cv2.INTER_LANCZOS4),
                     ("INTER_AREA", cv2.INTER_AREA)]:
    c, m = resize_composition_test(crop1, method=method)
    print("  %-20s композиция: %.2e · коммутативность: %.2e" % (name, c, m))

### Вопрос, на который отвечаем вслух

Почему композиция двух шагов ×1/2 не равна одному шагу ×1/4? И почему порядок уменьшения по осям влияет на результат?

---

## TODO 4 — согласованное уменьшение с антиалиасингом (≈ 10 минут)

Слайды 34, 48. **Антиалиасинг перед уменьшением**: перед прореживанием нужно сгладить кадр фильтром Гаусса с σ ≈ 0.3 шага.

Напишите `consistent_downsample(img, scale=2.0, sigma=None)` → уменьшенный кадр.

Скелет — три шага:

1. привести кадр к `float64`;
2. если `sigma` не задан — взять правило `σ ≈ 0.3 * scale`;
3. применить `cv2.GaussianBlur`, затем `cv2.resize` с `INTER_LINEAR`.

> **Подсказка.** Правило из лекции 6: σ ≈ 0.25–0.3 шага. Для ×2: σ ≈ 0.5–0.6.

In [ ]:
def consistent_downsample(img, scale=2.0, sigma=None):
    """Согласованное уменьшение: фильтр Гаусса перед прореживанием."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 5–7 строк: привести к float64; sigma по правилу 0.3 * scale; GaussianBlur; resize; return
    raise NotImplementedError("TODO 4")
    # ───────────────────────────────────────────────────────────


down_consistent = consistent_downsample(crop1, scale=2.0)
assert down_consistent.shape == (h // 2, w // 2), "размер должен быть в 2 раза меньше"
assert down_consistent.dtype == np.float64, "результат должен быть в float64"
print("TODO 4 ✔ · согласованное уменьшение ×2: форма %s" % (down_consistent.shape,))

img_f = crop1.astype(np.float64)
h_img, w_img = img_f.shape[:2]

no_filter = cv2.resize(img_f, (w_img // 2, h_img // 2), interpolation=cv2.INTER_LINEAR)
with_filter = consistent_downsample(img_f, scale=2.0)
inter_area = cv2.resize(img_f, (w_img // 2, h_img // 2), interpolation=cv2.INTER_AREA).astype(np.float64)

ref_small = cv2.resize(img_f, (w_img // 4, h_img // 4), interpolation=cv2.INTER_AREA).astype(np.float64)
ref_back = cv2.resize(ref_small, (w_img // 2, h_img // 2), interpolation=cv2.INTER_LINEAR)

print("\nКачество уменьшения ×2 (чем меньше ошибка к эталону, тем лучше):")
for name, result in [("без фильтра", no_filter),
                     ("Гаусс σ=0.6", with_filter),
                     ("INTER_AREA", inter_area)]:
    err = np.sqrt(np.mean((result - ref_back) ** 2))
    print("  %-20s RMSE к эталону = %.2f" % (name, err))

viz.grid({"без фильтра": no_filter / 255, "Гаусс σ=0.6": with_filter / 255,
          "INTER_AREA": inter_area / 255, "эталон (×4 → ×2)": ref_back / 255}, cols=4, size=2.8)

### Вопрос, на который отвечаем вслух

Почему `INTER_AREA` даёт лучший антиалиасинг, чем `INTER_LINEAR` при уменьшении? И почему согласованный ресайз важен для обучения нейросетей?

---

## Клиника: ДЗ 2 финал + мост к ДЗ 3 (последние 20 минут)

**ДЗ 2 «Сканер документов»** выдано на занятии 7 (вт 13.10), дедлайн уточнить у лектора.

**Что из занятия 10 идёт в ДЗ 2:**

| Тема занятия 10 | Где в ДЗ 2 | Правило |
|----------------|------------|---------|
| Гомография (4 точки) | выравнивание документа по 4 углам | `cv2.getPerspectiveTransform` или `findHomography`; порядок углов: ЛВ, ПВ, ПН, ЛН |
| `cv2.warpPerspective` | выпрямление документа | размер выхода из `--aspect` |
| Ресайз с антиалиасингом | уменьшение страниц перед показом | `INTER_AREA` или Гаусс σ ≈ 0.3 шага |
| Согласованность ресайза | аугментации при обучении (ДЗ 3) | цепочки масштабов должны быть согласованы |

**Мост к ДЗ 3 (панорамы):**

| Тема занятий 9–10 | Где в ДЗ 3 | Правило |
|-------------------|------------|---------|
| SIFT/ORB + matching | поиск соответствий между кадрами панорамы | `ratio=0.7–0.8` + `findHomography` с RANSAC |
| Гомография | выравнивание кадров панорамы | минимум 4 инлайнера после RANSAC |
| Склейка по полосам (L6) | бесшовная панорама | маска шва по перекрытию |
| Согласованный ресайз | подготовка кадров к склейке | `INTER_AREA` для уменьшения |

**Чек-лист перед сдачей ДЗ 2:**

- `pytest tests -q`;
- 4 угла найдены, гомография оценена, документ выпрямлен;
- `uint8`: все операции во `float32`, обратно через `np.clip`.

### Полный конвейер ДЗ 2: от углов к выровненному документу

In [ ]:
H_doc = compute_homography_dlt(dst_pts, src_pts)
doc_rectified = cv2.warpPerspective(crop2, H_doc, (w, h),
                                    borderMode=cv2.BORDER_REPLICATE)
err_doc = np.abs(crop1[20:-20, 20:-20] - doc_rectified[20:-20, 20:-20]).mean()
print("Конвейер выравнивания документа:")
print("  4 угла → гомография → выпрямление")
print("  ошибка: %.2f кодов (внутренняя часть)" % err_doc)
viz.grid({"повёрнутый документ": crop2 / 255, "выпрямленный": doc_rectified / 255}, cols=2, size=4)

### Вопросы клиники

Принесли вопрос по ДЗ 2 — задавайте сейчас. Если вопросов нет, разбираем мост к ДЗ 3 (панорамы).
С занятия 5 ассистент принимает **показ ДЗ**: одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка (по желанию, +0.5, до конца следующей недели)

**Полный конвейер панорамы**: два кадра → детектор → дескриптор → matching → RANSAC → гомография → склейка.

Напишите `build_panorama(img1, img2, detector='sift', ratio=0.75)` → склеенная панорама.

Сдаётся ноутбук с функцией и визуализацией панорамы.

In [ ]:
def build_panorama(img1, img2, detector='sift', ratio=0.75):
    """Полный конвейер панорамы: детектор → дескриптор → matching → RANSAC → гомография → склейка."""
    raise NotImplementedError("⭐ звёздочка: полный конвейер панорамы")


try:
    pano = build_panorama(crop1.astype(np.uint8), crop2.astype(np.uint8))
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть")
else:
    print("⭐ панорама: %d × %d" % (pano.shape[1], pano.shape[0]))
    viz.grid({"Панорама": pano / 255 if pano.max() > 1 else pano}, cols=1, size=6)

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–4 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **зачем нужны 4 пары точек для гомографии** и почему **стандартный ресайз не согласован**;
- [ ] на вопросы «почему цикл ×2 → ×1/2 не возвращает оригинал», «зачем антиалиасинг перед уменьшением» и «чем отличается аффинное от проективного» есть внятный ответ.

2 % итоговой оценки, ставится в классе. Досылать ничего не нужно. **Занятие 11 — модель камеры и калибровка.**